This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
#!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Text classification

이 장에서는 다음 내용을 다룹니다.

* **자연어 처리(NLP)** 분야 소개
* 텍스트 입력을 **숫자 입력으로 전처리**
* 간단한 **텍스트 분류 모델** 구축


---
이 장에서는 다음 두 장에서 다룰 텍스트 입력 처리의 기초를 다집니다. 이 장을 마치면 다양한 방법으로 간단한 텍스트 분류기를 만들 수 있게 될 것입니다. 이를 통해 다음 장에서 트랜스포머와 같은 더 복잡한 모델을 구축할 수 있는 기반을 마련할 수 있습니다.

### A brief history of natural language processing

컴퓨터 과학에서 우리는 영어나 중국어와 같**은 인간 언어를 "자연**어"라고 부릅니다. 이는 LISP, 어셈블리, XML처럼 기계를 위해 설계된 언어와 구별하기 위한 것입니다. 모든 기계어는 설계된 언어입니다. 엔지니어가 어떤 문장을 만들 수 있고 그 의미가 무엇인지 설명하는 형식적인 규칙들을 작성하는 것에서 시작됩니다. 규칙이 먼저 만들어졌고, 규칙 집합이 완성된 후에야 사람들이 언어를 사용하기 시작했습니다. 인간 언어는 정반대입니다. 사용이 먼저이고 규칙은 나중에 생겨납니다. 자연어는 생물체처럼 진화 과정을 통해 형성되었습니다. 이것이 자연어를 "자연어"라고 부르는 이유입니다. 영어 문법과 같은 자연어의 "규칙"은 나중에 형식화되었으며, 사용자들이 종종 무시하거나 어깁니다. 결과적으**로 기계가 읽을 수 있는 언어는 고도로 구조화되고 **엄격한 반면**, 자연어는 모호하고 혼란스럽고 방대하며 끊임없이 변화하는 지저분한 **언어입니다.

컴퓨터 과학자들은 오랫동안 자연어를 입력받거나 생성할 수 있는 시스템의 잠재력에 주목해 왔습니다. 언어, 특히 문자는 우리의 소통과 문화 생산의 대부분을 뒷받침합니다. 수 세기 동안 축적된 인류의 지식은 텍스트 형태로 저장되어 있으며, 인터넷 역시 대부분 텍스트로 이루어져 있고, 심지어 우리의 생각조차 언어에 기반하고 있습니다! 컴퓨터를 이용하여 언어를 해석하고 조작하는 기술을 자연어 처리(NLP)라고 합니다. NLP는 제2차 세계대전 직후 연구 분야로 처음 제안되었는데, 당시 일부 학자들은 언어 이해를 일종의 "암호 해독"으로 보고 자연어를 정보 전달에 사용되는 "암호"로 간주했습니다.

초창기 NLP 연구자들은 LISP처럼 영어의 "규칙 집합"을 글로 작성할 수 있을 것이라고 순진하게 생각했습니다. 1950년대 초, IBM과 조지타운 대학교의 연구원들은 러시아어를 영어로 번역하는 시스템을 시연했습니다. 이 시스템은 6개의 고정된 규칙으로 구성된 문법과 수백 개의 요소(단어와 접미사)로 이루어진 조회 테이블을 사용하여 엄선된 60개의 러시아어 문장을 정확하게 번역했습니다. 이 시스템의 목표는 기계 번역에 대한 관심과 투자를 유치하는 것이었고, 그러한 점에서 큰 성공을 거두었습니다. 시연의 한계에도 불구하고, 저자들은 5년 안에 번역 문제가 해결될 것이라고 주장했습니다. 이후 10년 가까이 연구 자금이 쏟아졌습니다. 그러나 그러한 시스템을 일반화하는 것은 엄청나게 어려운 일이었습니다. 단어는 문맥에 따라 의미가 극적으로 변하기 때문에, 어떤 문법 규칙이든 수많은 예외를 허용해야 했습니다. 몇 가지 엄선된 예시에서 뛰어난 성능을 보이는 프로그램을 개발하는 것은 비교적 간단했지만, 인간 번역가와 경쟁할 수 있는 견고한 시스템을 구축하는 것은 전혀 다른 문제였습니다. 10년 후, 영향력 있는 미국 보고서에서 이러한 진전 부족을 지적했고, 자금 지원은 중단되었습니다.

이러한 좌절과 기대와 실망의 반복적인 변동에도 불구하고, 수작업으로 만든 규칙은 1990년대까지 지배적인 접근 방식으로 남아 있었습니다. 문제점은 명백했지만, 문법을 설명하기 위해 기호 규칙을 작성하는 것 외에는 마땅한 대안이 없었습니다. 그러나 1980년대 후반, 더 빠른 컴퓨터와 더 많은 양의 데이터가 활용 가능해지면서 연구는 새로운 방향으로 나아가기 시작했습니다. 엔지니어라면 누구나 임의적인 규칙들을 마구잡이로 만들어 시스템을 구축하다 보면 "데이터 모음을 활용해서 이런 규칙들을 자동으로 찾아낼 수 있을까? 내가 직접 규칙을 만들어내는 대신, 규칙 공간 안에서 규칙을 검색할 수 있을까?"라는 질문을 던지게 될 겁니다. 그리고 바로 그 순간, 머신러닝이라는 영역으로 발을 들여놓게 되는 것이죠.

1980년대 후반, 자연어 처리에 머신러닝 접근 방식이 등장하기 시작했습니다. 초기에는 의사결정 트리를 기반으로 했는데, 그 목적은 하드코딩된 언어 시스템의 if/then/else 규칙과 같은 것을 자동화하는 것이었습니다. 그 후 로지스틱 회귀를 시작으로 통계적 접근 방식이 속도를 내기 시작했습니다. 시간이 흐르면서 학습된 매개변수 모델이 주를 이루었고, 언어학을 모델에 직접 적용하는 것은 오히려 방해가 된다는 의견도 나왔습니다. 초기 음성 인식 연구자였던 프레데릭 젤리넥은 1990년대에 "언어학자를 해고할 때마다 음성 인식기의 성능이 향상된다"고 농담하기도 했습니다.

컴퓨터 비전이 픽셀에 적용되는 패턴 인식인 것처럼, 현대 자연어 처리 분야는 텍스트 속 단어에 적용되는 패턴 인식입니다. 실용적인 응용 분야는 무궁무진합니다.

* 이메일 내용을 바탕으로 스팸일 확률은 얼마일까요? (텍스트 분류)
* 영어 문장을 바탕으로 가장 가능성이 높은 러시아어 번역은 무엇일까요? (번역)
* 불완전한 문장이 주어졌을 때, 다음에 어떤 단어가 올 가능성이 가장 높을까요? (언어 모델링)

이 책에서 학습할 텍스트 처리 모델은 인간과 같은 언어 이해 능력을 갖추지는 않습니다. 오히려 입력 데이터에서 통계적 규칙성을 찾는 데 그치는데, 이것만으로도 다양한 실제 작업에서 뛰어난 성능을 발휘하기에 충분합니다.

지난 10년간 자연어 처리(NLP) 연구자와 실무자들은 텍스트에 대한 구체적인 통계적 질문에 대한 답을 학습하는 것이 얼마나 놀라운 효과를 가져오는지 발견했습니다. 2010년대에 연구자들은 LSTM 모델을 텍스트에 적용하기 시작했고, 이로 인해 NLP 모델의 매개변수 개수와 학습에 필요한 컴퓨팅 자원이 크게 증가했습니다. 결과는 고무적이었습니다. LSTM은 이전 방식보다 훨씬 높은 정확도로 미지의 예시에 일반화할 수 있었지만, 결국 한계에 부딪혔습니다. LSTM은 문장과 단락이 많은 긴 텍스트에서 의존 관계를 추적하는 데 어려움을 겪었고, 컴퓨터 비전 모델에 비해 학습 속도가 느리고 다루기 어려웠습니다.

2010년대 후반, 구글 연구진은 LSTM의 확장성 문제를 해결하는 새로운 아키텍처인 트랜스포머(Transformer)를 발견했습니다. 모델 크기와 학습 데이터의 크기를 함께 늘리면 트랜스포머는 점점 더 정확한 성능을 보였습니다. 더 나아가, 트랜스포머 학습에 필요한 연산은 긴 시퀀스에서도 효과적으로 병렬화할 수 있었습니다. 학습에 투입되는 머신의 수를 두 배로 늘리면 결과를 기다리는 시간을 절반으로 줄일 수 있었습니다.

트랜스포머 아키텍처의 발견과 더불어 점점 빨라지는 GPU 및 CPU의 등장으로 지난 몇 년간 자연어 처리(NLP) 모델에 대한 투자와 관심이 폭발적으로 증가했습니다. ChatGPT와 같은 채팅 시스템은 임의의 주제와 질문에 대해 유창하고 자연스러운 텍스트를 생성하는 능력으로 대중의 관심을 사로잡았습니다. 이러한 모델을 학습하는 데 사용되는 원문은 인터넷에 존재하는 모든 텍스트의 상당 부분을 차지하며, 개별 모델을 학습하는 데 드는 컴퓨팅 비용은 수천만 달러에 달할 수 있습니다. 하지만 이러한 기술에 대한 과장된 기대는 어느 정도 현실을 직시해야 합니다. 결국 이러한 기술들은 패턴 인식에 불과합니다. 인간은 '말하는 사물'에서 지능을 찾으려는 경향이 있지만, 이러한 모델들은 인간의 지능과는 완전히 다른 (그리고 훨씬 비효율적인!) 방식으로 훈련 데이터를 복제하고 합성합니다. 그러나 '빠진 단어 맞추기'와 같은 매우 단순한 훈련 환경에서 복잡한 행동이 나타나는 것은 지난 10년간 기계 학습 분야에서 가장 놀라운 실증적 결과 중 하나라고 할 수 있습니다.

다음 세 장에서는 텍스트 데이터를 활용한 다양한 기계 학습 기법을 살펴보겠습니다. 1990년대까지 널리 사용되었던 하드코딩된 언어적 특징에 대한 논의는 생략하고, 텍스트 분류를 위한 로지스틱 회귀 분석부터 기계 번역을 위한 LSTM 훈련까지 다양한 기법을 다룰 것입니다. 특히 트랜스포머 모델을 자세히 살펴보고, 텍스트 영역에서 뛰어난 확장성과 일반화 능력을 갖추게 된 요인을 분석할 것입니다. 자, 시작해 볼까요?

### Preparing text data

영어 문장 하나를 살펴보겠습니다.
```
The quick brown fox jumped over the lazy dog.
```
이전 장에서 다룬 딥러닝 기법들을 적용하기 전에 해결해야 할 분명한 문제가 있습니다. 바로 입력 데이터가 숫자가 아니라는 점입니다! 모델링을 시작하기 전에 **먼저 텍스트를 숫자 텐서로 변환**해야 합니다. 이미지처럼 비교적 자연스러운 숫자 표현이 가능한 것과는 달리, 텍스트의 숫자 표현은 여러 가지 방법으로 구축할 수 있습니다.

간단한 방법은 표준 텍스트 파일 형식을 차용하여 ASCII 인코딩과 같은 방식을 사용하는 것입니다. 입력 데이터를 일련의 문자로 나누고 각 문자에 고유한 인덱스를 할당할 수 있습니다. 또 다른 직관적인 방법은 단어를 기반으로 표현을 구축하는 것입니다. 먼저 문장을 공백과 구두점을 기준으로 분리한 다음, 각 단어를 고유한 숫자 표현으로 매핑하는 방식입니다.

이 두 가지 방법 모두 시도해 볼 만한 좋은 방법이며, 일반적으로 모든 텍스트 전처리 과정에는 **텍스트를 토큰이라고 하는 작은 단위로 분할하는 단계**가 포함됩니다. 텍스트를 분할하는 강력한 도구는 정규 표현식입니다. 정규 표현식은 텍스트에서 특정 문자 패턴을 유연하게 일치시킬 수 있습니다.

이제 정규 표현식을 사용하여 문자열을 일련의 문자로 분할하는 방법을 살펴보겠습니다. 가장 기본적인 정규 표현식은 "."이며, 이는 입력 텍스트의 모든 문자와 일치합니다.

In [24]:
import regex as re

def split_chars(text):
    return re.findall(r".", text)

이제 이 함수를 예시 입력 문자열에 적용해 보겠습니다.

In [27]:
chars = split_chars("The quick brown fox jumped over the lazy dog.")
chars[:12]

['T', 'h', 'e', ' ', 'q', 'u', 'i', 'c', 'k', ' ', 'b', 'r']

정규 표현식(Regex)을 사용하면 텍스트를 단어 단위로 쉽게 분리할 수 있습니다. **"[\w]+" 정규 표현식은 연속된 공백이 아닌 문자를 선택**하고, **"[.,!?;]"는 괄호 안의 문장 부호를 선택**합니다. 이 두 가지를 결합하여 각 단어와 문장 부호를 토큰으로 분리하는 정규 표현식을 만들 수 있습니다. 

In [31]:
def split_words(text):
    return re.findall(r"[\w]+|[.,!?;]", text)

다음은 테스트 문장에 대한 결과입니다.

In [34]:
split_words("The quick brown fox jumped over the dog.")

['The', 'quick', 'brown', 'fox', 'jumped', 'over', 'the', 'dog', '.']

문자열 분할을 통해 단일 문자열을 토큰 시퀀스로 만들지만, 여전히 **문자열 토큰을 숫자 입력으로 변환**해야 합니다. 가장 일반적인 접근 방식은 **각 토큰을 고유한 정수 인덱스에 매핑**하는 것인데, 이를 **입력 인덱싱**이라고 합니다. 이는 토큰화된 입력을 유연하고 가역적으로 표현할 수 있는 방법으로, 다양한 모델링 접근 방식과 호환됩니다. 이후에는 토큰 인덱스를 모델이 입력받는 잠재 공간으로 어떻게 매핑할지 결정할 수 있습니다.

문자 토큰의 경우 ASCII 조회를 사용하여 각 토큰의 인덱스를 지정할 수 있습니다. 예를 들어, ord('A') → 65, ord('z') → 122와 같습니다. 그러나 유니코드 사양에는 백만 개가 넘는 문자가 있으므로 다른 언어를 고려하기 시작하면 이 방법은 확장성이 떨어집니다. 보다 강력한 기술은 훈련 데이터의 특정 토큰을 우리가 관심 있는 데이터(NLP에서는 어휘라고 함)에 나타나는 인덱스로 매핑하는 것입니다. 이 방법은 단어 수준 토큰뿐만 아니라 문자 수준 토큰에도 쉽게 적용할 수 있다는 장점이 있습니다.

어휘를 사용하여 텍스트를 변환하는 방법을 살펴보겠습니다. **토큰을 인덱스에 매핑하는 간단한 파이썬 딕셔너리를 만들고, 입력을 토큰 단위로 분할한 다음, 토큰에 인덱싱을 적용**해 보겠습니다.

In [49]:
vocabulary = {
    "[UNK]": 0,
    "the": 1,
    "quick": 2,
    "brown": 3,
    "fox": 4,
    "jumped": 5,
    "over": 6,
    "dog": 7,
    ".": 8,
}
words = split_words("The quick brown fox jumped over the lazy dog.")
indices = [vocabulary.get(word, 0) for word in words]
indices

[0, 2, 3, 4, 5, 6, 1, 0, 7, 8]

우리는 어휘에 "[UNK]"라는 특수 토큰을 도입합니다. **이 토큰은 어휘에 없는 단어**를 나타냅니다. 이렇게 하면 테스트 데이터에만 나타나는 단어까지 포함하여 모든 입력값을 인덱싱할 수 있습니다. **앞의 예에서 "lazy"는 어휘에 포함되지 않았으므로 "[UNK]" 인덱스 0**에 해당합니다.

이러한 간단한 텍스트 변환을 통해 텍스트 전처리 파이프라인 구축의 첫걸음을 내딛을 수 있습니다. 하지만 고려해야 할 또 다른 일반적인 텍스트 조작 유형이 있는데, 바로 **표준화**입니다.

다음 두 문장을 살펴보세요.

* “sunset came. i was staring at the Mexico sky. Isn’t nature splendid??”

* “Sunset came; I stared at the México sky. Isn’t nature splendid?”

둘은 매우 유사하며, 사실상 거의 동일합니다. 하지만 앞서 설명한 대로 인덱스로 변환하면 "i"와 "I"는 서로 다른 문자이고, "Mexico"와 "México"는 서로 다른 단어이며, "isnt"는 "isn't"와 다르기 때문에 매우 다른 표현이 나옵니다. 머신 러닝 모델은 "i"와 "I"가 같은 문자라는 것, "é"가 악센트가 있는 "e"라는 것, 또는 "staring"과 "stared"가 같은 동사의 두 가지 형태라는 것을 사전에 알지 못합니다. 텍스트 표준화는 모델이 처리해야 할 불필요한 인코딩 차이를 제거하는 것을 목표로 하는 기본적인 특징 엔지니어링 기법입니다. 이는 머신 러닝에만 국한된 것이 아니라 검색 엔진을 구축할 때도 마찬가지입니다.

간단하고 널리 사용되는 표준화 방법 중 하나는 소문자로 변환하고 구두점을 제거하는 것입니다. 두 문장을 다음과 같이 바꿀 수 있습니다.

* “sunset came i was staring at the mexico sky isn't nature splendid”
* “sunset came i stared at the méxico sky isn't nature splendid”

훨씬 더 가까워졌네요. 모든 글자의 악센트 기호를 제거하면 더 비슷해질 수 있습니다.

표준화를 통해 할 수 있는 일은 매우 많으며, 과거에는 모델 성능 향상에 있어 가장 중요한 영역 중 하나였습니다. 수십 년 동안 자연어 처리(NLP) 분야에서는 정규 표현식을 사용하여 단어를 공통 어근(예: "tired" → "tire", "trophies" → "trophy")으로 매핑하는 것이 일반적이었는데, 이를 어간 추출 또는 표제어 추출이라고 합니다. 그러나 모델의 표현력이 향상됨에 따라 이러한 유형의 표준화는 오히려 역효과를 낳는 경향이 있습니다. 단어의 시제와 복수형은 의미를 나타내는 데 필수적인 신호입니다. 오늘날 사용되는 대규모 모델의 경우, 대부분의 표준화는 가능한 한 가볍게 수행됩니다. 예를 들어, 추가 처리를 하기 전에 모든 입력을 표준 문자 인코딩으로 변환하는 방식입니다.

표준화를 통해 텍스트 전처리에는 이제 세 가지 단계가 있습니다(그림 14.1).

1. **표준화** - 기본적인 텍스트 변환을 통해 입력을 정규화합니다.
2. **분할** - 텍스트를 토큰 시퀀스로 분할합니다.
3. **인덱싱** - 어휘집을 사용하여 토큰을 인덱스에 매핑합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch14/text-pipeline.c09bbad6.png" width="300"><br>Figure 14.1: The text preprocessing pipeline</p>

사람들은 흔히 **전체 과정을 토큰화**라고 부르고, 텍스트를 토큰 인덱스 시퀀스로 매핑하는 객체를 토크나이저라고 부릅니다. 이제 몇 가지 토크나이저를 만들어 보겠습니다.

#### Character and word tokenization

먼저 입력 문자열의 각 문자를 정수로 매핑하는 **문자 수준 토크나이저**를 만들어 보겠습니다. 간단하게 하기 위해 한 가지 표준화 단계만 사용하겠습니다. 모든 입력을 소문자로 변환하는 것입니다.

In [55]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r".", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

꽤 간단합니다. 이 기능을 사용하기 전에 입력 텍스트를 기반으로 토큰 어휘를 계산하는 함수를 먼저 만들어야 합니다. 모든 문자를 고유한 인덱스에 매핑하는 대신, 입력 데이터에서 가장 자주 사용되는 토큰만으로 어휘 크기를 제한할 수 있도록 해 보겠습니다. 모델링 단계에서 어휘 크기를 제한하는 것은 모델의 매개변수 개수를 줄이는 중요한 방법이 될 것입니다.

In [58]:
import collections

def compute_char_vocabulary(inputs, max_size):
    char_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

이제 **단어 수준 토크나이저**에도 동일한 작업을 수행할 수 있습니다. 문자 수준 토크나이저와 동일한 코드를 사용하되, 분할 단계를 다르게 하면 됩니다.

In [61]:
class WordTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r"[\w]+|[.,!?;]", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

이 새로운 분할 규칙을 어휘 함수에 대입할 수도 있습니다.

In [64]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r"[\w]+|[.,!?;]", x)
        word_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = word_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

이제 실제 입력 데이터인 허먼 멜빌의 소설 『모비딕』 전문을 사용하여 토크나이저를 테스트해 보겠습니다. 먼저 두 토크나이저에 대한 어휘집을 구축한 다음, 이를 사용하여 텍스트를 토큰화해 보겠습니다.

In [69]:
import keras

filename = keras.utils.get_file(
    origin="https://www.gutenberg.org/files/2701/2701-0.txt",
)
moby_dick = list(open(filename, "r", encoding="utf-8"))

vocabulary = compute_char_vocabulary(moby_dick, max_size=100)
char_tokenizer = CharTokenizer(vocabulary)

문자 단위 토크나이저가 계산한 내용을 살펴보겠습니다.

In [72]:
print("Vocabulary length:", len(vocabulary))

Vocabulary length: 73


In [74]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

Vocabulary start: ['[UNK]', ' ', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h']


In [76]:
print("Vocabulary end:", list(vocabulary.keys())[-10:])

Vocabulary end: ['\u200f', 'ח', 'ו', '\u200e', 'ϰ', 'η', 'τ', 'ο', 'ς', 'â']


In [78]:
print("Line length:", len(char_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 63


그렇다면 단어 수준 토크나이저는 어떨까요?

In [81]:
vocabulary = compute_word_vocabulary(moby_dick, max_size=2_000)
word_tokenizer = WordTokenizer(vocabulary)

단어 수준 토크나이저에 대해서도 동일한 데이터를 출력할 수 있습니다.

In [84]:
print("Vocabulary length:", len(vocabulary))

Vocabulary length: 2000


In [86]:
print("Vocabulary start:", list(vocabulary.keys())[:5])

Vocabulary start: ['[UNK]', ',', 'the', '.', 'of']


In [88]:
print("Vocabulary end:", list(vocabulary.keys())[-5:])

Vocabulary end: ['beholding', 'hearse', 'mount', 'tackle', 'objects']


In [90]:
print("Line length:", len(word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 13


두 가지 토큰화 기법의 장단점을 이미 확인할 수 있습니다. **문자 수준 토큰화기는 책 전체를 처리하는 데 64개의 어휘 용어만 필요**하지만, **각 입력값을 매우 긴 시퀀스로 인코딩**합니다. **단어 수준 토큰화기는 2,000개 용어의 어휘**를 빠르게 채울 수 있지만(책에 나오는 모든 단어를 색인화하려면 17,000개의 용어가 있는 사전이 필요합니다!), **출력값은 훨씬 짧습니다**.

머신러닝 전문가들이 점점 더 많은 데이터와 매개변수를 사용하여 모델을 확장함에 따라 단어 수준 토큰화와 문자 수준 토큰화 모두의 단점이 분명해졌습니다. **단어 수준 토큰화가 제공하는 "압축"은 매우 중요한 장점**으로 작용하여 모델에 더 긴 시퀀스를 입력할 수 있게 해줍니다. 그러나 오늘**날 수조 개의 단어로 구성된 대규모 데이터셋을 위해 단어 수준 어휘를 구축하려고 하**수억 개의 용어로 이루어진 비효율적인 어휘**를 얻게 됩니다. 단어 수준 어휘 크기를 지나치게 제한하면 많은 텍스트가 "[UNK]" 토큰으로 인코딩되어 중요한 정보가 손실될 수 있습니다.

이러한 문제로 인해 단어 수준과 문자 수준 접근 방식 사이의 간극을 메우려는 세 번째 유형의 토큰화 방식인 **서브워드 토큰화가 인기**를 얻고 있습니다.

#### Subword tokenization

서브워드 토큰화는 문자 수준과 단어 수준 인코딩 기법의 장점을 결합하는 것을 목표로 합니다. WordTokenizer의 간결한 출력 생성 능력과 CharTokenizer의 적은 어휘로 다양한 입력을 인코딩하는 능력을 모두 활용하고자 합니다.

이상적인 토크나이저를 찾는 과정을 입력 데이터의 이상적인 압축을 찾는 과정으로 생각할 수 있습니다. 토큰 길이를 줄이면 전체 예제의 길이가 압축됩니다. 어휘가 작으면 각 토큰을 표현하는 데 필요한 바이트 수가 줄어듭니다. 이 두 가지를 모두 달성하면 짧고 정보가 풍부한 시퀀스를 딥러닝 모델에 입력할 수 있습니다.

압축과 토큰화 사이의 이러한 유사성은 처음에는 명확하지 않았지만, 강력한 연관성을 보여줍니다. 지난 10년간 자연어 처리 연구에서 가장 실용적인 방법 중 하나는 199**0년대 무손실 압축 알고리즘인 바이트 쌍(byte-pair encoding)** 인코딩[1]을 토큰화에 재활용한 것입니다. 이 알고리즘은 ChatGPT를 비롯한 많은 모델에서 현재까지 사용되고 있습니다. 이 섹션byte-pair encoding 인코딩 알고리즘을 사용하는 토크나이저를 구축합니다.

바이트 쌍 인코딩의 기본 **아이디어는 기본적인 문자 어휘에서 시작하여 공통적인 쌍들을 점진적으로 더 긴 문자 시퀀스로 "병**합"하는 것입니다. 다음 입력 텍스트에서 시작한다고 가정해 보겠습니다.

In [96]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

WordTokenizer와 마찬가지로, 먼저 텍스트에 있는 모든 단어의 개수를 계산합니다. 단어 개수 사전을 만들 때, 텍스트를 문자 단위로 나누고 각 문자를 공백으로 연결합니다. 이렇게 하면 다음 단계에서 문자 쌍을 더 쉽게 비교할 수 있습니다.

In [120]:
def count_and_split_words(data):
    counts = collections.Counter()
    for line in data:
        line = line.lower()
        for word in re.findall(r"[\w]+|[.,!?;]", line):
            chars = re.findall(r".", word)
            split_word = " ".join(chars)
            counts[split_word] += 1
    return dict(counts)

counts = count_and_split_words(data)

우리 데이터에 적용해 보겠습니다.

In [123]:
counts

{'t h e': 3,
 'q u i c k': 2,
 'b r o w n': 3,
 'f o x': 2,
 's l o w': 1,
 'f o x h o u n d': 1}

분할된 단어 개수에 바이트 쌍 인코딩을 적용하려면 두 문자를 찾아 새로운 기호로 병합합니다. 모든 단어의 모든 문자 쌍을 고려하고 가장 빈번하게 나타나는 쌍만 병합합니다. 앞의 예에서 가장 빈번한 문자 쌍은 "brown"(데이터에 세 번 나타남)과 "slow"(한 번 나타남) 두 단어 모두에서 나타나는 ("o", "w")입니다. 이 쌍을 결합하여 "ow"라는 새로운 기호를 만들고 "o w"가 나타나는 모든 경우를 병합합니다.

이후에도 쌍을 세고 병합하는 과정을 반복합니다. 이제 "ow"는 하나의 단위로 간주되어 예를 들어 "l"과 병합되어 "low"를 형성할 수 있습니다. 가장 빈번하게 나타나는 기호 쌍을 점진적으로 병합함으로써 점점 더 큰 하위 단어들의 어휘를 구축할 수 있습니다.

이 과정을 예시 데이터셋에 적용해 보겠습니다.

In [126]:
def count_pairs(counts):
    pairs = collections.Counter()
    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

def merge_pair(counts, first, second):
    split = re.compile(rf"(?<!\S){first} {second}(?!\S)")
    merged = f"{first}{second}"
    return {split.sub(merged, word): count for word, count in counts.items()}

for i in range(10):
    pairs = count_pairs(counts)
    if not pairs:
        print(f"{i}번째 반복에서 병합 종료")
        break
    first, second = max(pairs, key=pairs.get)
    counts = merge_pair(counts, first, second)
    print(list(counts.keys()))

['t h e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['th e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'br ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'fo x', 's l ow', 'fo x h o u n d']
['the', 'q u i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qu i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qui c k', 'brown', 'fox', 's l ow', 'fox h o u n d']


우리가 얻는 결과는 위와 같습니다.

**자주 사용되는 단어는 완전히 병합**되는 반면, **덜 자주 사용되는 단어는 부분적으로만 병합**되는 것을 확인할 수 있습니다.

이제 이를 확장하여 바이트 쌍 인코딩(byte-pair encoding) 어휘를 계산하는 완전한 함수를 만들 수 있습니다. 입력 텍스트에 있는 모든 문자를 어휘로 시작하고, 원하는 길이에 도달할 때까지 병합된 기호(점점 더 큰 하위 단어)를 점진적으로 추가합니다. 또한 병합 규칙과 적용 순서를 포함하는 별도의 사전을 유지합니다. 다음으로, 이러한 병합 규칙을 사용하여 새로운 입력 텍스트를 토큰화하는 방법을 살펴보겠습니다.

In [130]:
def compute_sub_word_vocabulary(dataset, vocab_size):
    counts = count_and_split_words(dataset)

    char_counts = collections.Counter()
    for word in counts:
        for char in word.split():
            char_counts[char] += counts[word]
    most_common = char_counts.most_common()
    vocab = ["[UNK]"] + [char for char, freq in most_common]
    merges = []

    while len(vocab) < vocab_size:
        pairs = count_pairs(counts)
        if not pairs:
            break
        first, second = max(pairs, key=pairs.get)
        counts = merge_pair(counts, first, second)
        vocab.append(f"{first}{second}")
        merges.append(f"{first} {second}")

    vocab = dict((token, index) for index, token in enumerate(vocab))
    merges = dict((token, rank) for rank, token in enumerate(merges))
    return vocab, merges

이제 병합 규칙을 적용하여 새로운 입력 텍스트를 토큰화하는 SubWordTokenizer를 만들어 보겠습니다. standardize()와 index() 단계는 WordTokenizer와 동일하게 유지하고, 모든 변경 사항은 split() 메서드에 적용합니다.

splitting 단계에서는 먼저 모든 입력을 단어로 분할하고, 각 단어를 문자로 분할한 다음, 학습된 병합 규칙을 분할된 문자에 적용합니다. 이렇게 하면 서브워드(subword)가 남게 되는데, 이 토큰은 학습 데이터에서 입력 단어의 빈도에 따라 전체 단어, 부분 단어 또는 단순 문자가 될 수 있습니다. 이러한 서브워드가 출력 토큰이 됩니다.

In [138]:
class SubWordTokenizer:
    def __init__(self, vocabulary, merges):
        self.vocabulary = vocabulary
        self.merges = merges
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def bpe_merge(self, word):
        while True:
            pairs = re.findall(r"(?<!\S)\S+ \S+(?!\S)", word, overlapped=True)
            if not pairs:
                break
            best = min(pairs, key=lambda pair: self.merges.get(pair, 1e9))
            if best not in self.merges:
                break
            first, second = best.split()
            split = re.compile(rf"(?<!\S){first} {second}(?!\S)")
            merged = f"{first}{second}"
            word = split.sub(merged, word)
        return word

    def split(self, inputs):
        tokens = []
        for word in re.findall(r"[\w]+|[.,!?;]", inputs):
            word = " ".join(re.findall(r".", word))
            word = self.bpe_merge(word)
            tokens.extend(word.split())
        return tokens

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

모비딕 전문에 토크나이저를 적용해 보겠습니다.

In [140]:
vocabulary, merges = compute_sub_word_vocabulary(moby_dick, 2_000)
sub_word_tokenizer = SubWordTokenizer(vocabulary, merges)

Words Tokenizer와 Char Tokenizer에서 했던 것처럼, 우리가 가진 어휘를 살펴보고 토크나이저에서 테스트 문장을 실행해 볼 수 있습니다.

In [142]:
print("Vocabulary length:", len(vocabulary))

Vocabulary length: 2000


In [143]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

Vocabulary start: ['[UNK]', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h', 'r']


In [144]:
print("Vocabulary end:", list(vocabulary.keys())[-7:])

Vocabulary end: ['stret', 'breath', 'na', 'thern', 'armed', 'fishermen', 'bright']


In [145]:
print("Line length:", len(sub_word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 16


SubWordTokenizer는 WordTokenizer보다 테스트 문장에서 토큰 길이가 약간 더 길지만(16개 대 13개), WordTokenizer와 달리 "[UNK]" 토큰을 사용하지 않고도 『모비딕』의 모든 단어를 토큰화할 수 있습니다. 어휘집에는 소스 텍스트의 모든 문자가 포함되어 있으므로 최악의 성능은 단어를 개별 문자로 토큰화하는 경우에 발생합니다. 작은 어휘집을 사용하면서도 드문 단어를 처리하여 평균 토큰 길이를 짧게 유지할 수 있었습니다. 이것이 바로 부분 단어 토큰화기의 장점입니다.

이 코드를 실행하는 속도가 단어 및 문자 토큰화기보다 눈에 띄게 느리다는 것을 알 수 있을 것입니다. 레퍼런스 하드웨어에서 약 1분이 소요됩니다. 병합 규칙을 학습하는 것은 입력 데이터 세트의 단어 수를 세는 것보다 훨씬 복잡합니다. 이는 부분 단어 토큰화의 단점이지만, 실제로는 중요한 문제가 되는 경우는 드뭅니다. 모델당 한 번만 어휘집을 학습하면 되며, 부분 단어 어휘집을 학습하는 비용은 일반적으로 모델 학습 비용에 비해 무시할 수 있을 정도로 작습니다.

이제 입력을 토큰화하는 세 가지 접근 방식을 살펴보았습니다. 이제 텍스트를 숫자 입력으로 변환할 수 있게 되었으니, 모델 학습으로 넘어가 보겠습니다.

토큰화에 대해 마지막으로 한 가지 더 말씀드리자면, 토크나이저의 작동 원리를 이해하는 것은 매우 중요하지만, 직접 토크나이저를 만들어야 하는 경우는 **드뭅니다. Keras를 비롯한 대부분의 딥러닝 프레임워크는 텍스트 입력 토큰화를 위한 유틸**리티를 제공합니다. 따라서 이 장의 나머지 부분에서는 Keras의 내장 기능을 활용할 것입니다.

**어떤 토큰화 기법을 사용해야 할까요?**

새로운 텍스트 모델링 문제를 접할 때 가장 먼저 답해야 할 질문 중 하나는 입력 데이터를 어떻게 토큰화할 것인가입니다. 이 장의 끝부분에서 살펴보겠지만, 사전 학습된 모델의 경우에는 이 질문이 간단합니다. 사전 학습에 사용된 토큰화 방식을 그대로 유지하거나, 모델 가중치에 포함된 입력 토큰의 유용한 표현을 버리면 됩니다.

하지만 처음부터 모델을 구축하는 경우에는 문제에 맞게 토큰화 방식을 조정할 수 있습니다. 일반적으로 단어 또는 부분 단어 토큰화 방식이 제공하는 압축률은 매우 중요합니다. 입력 데이터의 평균 길이가 짧을수록 모델은 텍스트 내의 장거리 의존 관계를 더 잘 추적하여 전반적인 성능을 향상시킬 수 있습니다. 이러한 이유로 부분 단어 토큰화 방식이 현대 언어 모델에서 가장 널리 사용되고 있습니다. 부분 단어 토큰화 방식은 일반적인 입력 데이터의 토큰 길이를 늘리지 않고도 드물거나 오타가 있는 단어를 처리할 수 있습니다.

하지만 모든 상황에 맞는 만능 해결책은 없습니다. 맞춤법 교정과 같은 자연어 처리(NLP)의 일부 문제는 입력 텍스트에 대한 저수준 문자 토큰화를 통해 이점을 얻을 수 있습니다. 반면, 단어 수준 접근 방식은 다루기 쉽고 이해하기도 간편합니다. 각 모델 입력은 사람이 읽는 단어에 해당하기 때문입니다. 따라서 예측에 대한 중요도에 따라 토큰 순위를 매기는 것이 쉽게 해석될 수 있습니다.

이 책의 본문에서는 세 가지 유형의 토크나이저를 모두 사용할 것입니다.

### Sets vs. sequences

머신러닝 모델이 개별 토큰을 어떻게 표현해야 하는지는 비교적 논란의 여지가 없는 문제입니다. 토큰은 범주형 특징(미리 정의된 집합의 값)이므로, 이를 처리하는 방법은 이미 알려져 있습니다. 토큰은 특징 공간의 차원이나 범주 벡터(이 경우에는 토큰 벡터)로 인코딩될 수 있습니다. 하지만 텍스트에서 토큰의 순서를 어떻게 인코딩해야 하는지는 훨씬 더 어려운 문제입니다.

자연어에서 순서 문제는 흥미로운 주제입니다. 시계열 데이터의 단계와는 달리, 문장 속 단어들은 자연스럽고 정형화된 순서를 가지고 있지 않습니다. 언어마다 비슷한 단어들을 배열하는 방식이 매우 다릅니다. 예를 들어, 영어의 문장 구조는 일본어와 상당히 다릅니다. 심지어 같은 언어 내에서도 단어 순서를 조금만 바꿔도 같은 의미를 전달할 수 있는 경우가 많습니다. 짧은 문장에서 단어들을 완전히 무작위로 배열하더라도 의미를 파악할 수는 있지만, 많은 경우 상당한 모호성이 발생합니다. 순서는 분명 중요하지만, 의미와의 관계는 단순하지 않습니다.

단어 순서를 어떻게 표현할 것인가는 다양한 자연어 처리(NLP) 아키텍처의 핵심 질문입니다. 가장 간단한 방법은 순서를 무시하고 텍스트를 순서가 없는 단어 집합으로 처리하는 것입니다. 이것이 바로 '단어 모음(bag-of-words)' 모델입니다. 또는 단어를 시계열 데이터의 단계처럼 순서대로 하나씩 처리하는 방법도 있습니다. 이 경우 이전 장에서 다룬 순환 신경망(RNN) 모델을 사용할 수 있습니다. 마지막으로, 하이브리드 접근 방식도 가능합니다. 트랜스포머(Transformer) 아키텍처는 기술적으로 순서에 구애받지 않지만, 처리하는 표현에 단어 위치 정보를 주입하여 문장의 여러 부분을 동시에 살펴볼 수 있습니다(RNN과는 달리). 단어 순서를 고려하기 때문에 RNN과 트랜스포머 모두 '순서 모델(sequence model)'이라고 불립니다.

역사적으로, 초기 NLP 분야의 머신러닝 응용 프로그램들은 대부분 순서 데이터를 무시한 '단어 모음' 모델에 그쳤습니다. 순서 모델에 대한 관심은 2015년 RNN의 재조명과 함께 다시 높아지기 시작했습니다. 오늘날 두 접근 방식 모두 여전히 유효합니다. 각 방법이 어떻게 작동하는지, 그리고 언제 어떤 방법을 사용해야 하는지 살펴보겠습니다.

잘 알려진 텍스트 분류 벤치마크인 IMDb 영화 리뷰 감정 분류 데이터셋을 사용하여 각 접근 방식을 시연해 보겠습니다. 4장과 5장에서는 벡터화된 IMDb 데이터셋을 사용했지만, 이제 실제 텍스트 분류 문제를 해결할 때처럼 원시 IMDb 텍스트 데이터를 직접 처리해 보겠습니다.

#### Loading the IMDb classification dataset

먼저 데이터셋을 다운로드하고 압축을 풀어보겠습니다.

In [ ]:
import os, pathlib, shutil, random

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"

디렉토리 구조를 나열해 보겠습니다.

In [ ]:
for path in imdb_extract_dir.glob("*/*"):
    if path.is_dir():
        print(path)

훈련 세트와 테스트 세트 모두에 긍정적인 예시와 부정적인 예시가 포함되어 있는 것을 볼 수 있습니다. IMDb 사이트에서 사용자 평점이 낮은 영화 리뷰는 neg/ 디렉토리에, 평점이 높은 리뷰는 pos/ 디렉토리에 분류되어 있습니다. 또한 unsup/ 디렉토리도 있는데, 이는 비지도 학습(unsupervised)을 의미합니다. 이 디렉토리에는 데이터셋 제작자가 의도적으로 레이블을 지정하지 않은 리뷰들이 저장되어 있으며, 긍정적인 리뷰일 수도 있고 부정적인 리뷰일 수도 있습니다.

이제 몇 가지 텍스트 파일의 내용을 살펴보겠습니다. 텍스트 데이터든 이미지 데이터든 모델링을 시작하기 전에 데이터의 형태를 확인하는 것이 중요합니다. 이를 통해 모델이 실제로 어떻게 작동하는지에 대한 직관을 더욱 확고히 할 수 있습니다.

In [ ]:
print(open(imdb_extract_dir / "train" / "pos" / "4077_10.txt", "r").read())

입력 텍스트를 토큰화하기 전에 몇 가지 중요한 수정 사항을 적용하여 학습 데이터의 사본을 만들겠습니다. 비지도 학습 리뷰는 일단 제외하고, 학습 중 정확도를 모니터링하기 위한 별도의 검증 세트를 생성합니다. 이를 위해 학습 텍스트 파일의 20%를 새 디렉터리로 분할합니다.

In [ ]:
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(val_dir / category)
    for file in src_files[:num_val_samples]:
        shutil.copy(src_dir / file, val_dir / category / file)
    os.makedirs(train_dir / category)
    for file in src_files[num_val_samples:]:
        shutil.copy(src_dir / file, train_dir / category / file)

이제 데이터를 로드할 준비가 되었습니다. 8장에서 `image_dataset_from_directory` 유틸리티를 사용하여 디렉토리 구조에 맞는 이미지와 레이블로 구성된 데이터셋을 생성했던 것을 기억하시나요? `text_dataset_from_directory` 유틸리티를 사용하면 텍스트 파일에 대해서도 똑같은 작업을 수행할 수 있습니다. 학습, 검증, 테스트를 위한 세 개의 데이터셋 객체를 생성해 보겠습니다.

In [ ]:
from keras.utils import text_dataset_from_directory

batch_size = 32
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

원래 훈련용 데이터와 테스트용 데이터가 각각 25,000개였는데, 검증용으로 분할한 후에는 훈련용 리뷰가 20,000개, 검증용 리뷰가 5,000개 남았습니다. 이 데이터를 활용해서 뭔가 학습해 보도록 하겠습니다.

### Set models

텍스트에서 토큰의 순서에 대한 가장 간단한 접근 방식은 순서를 무시하는 것입니다. 입력 리뷰를 토큰 ID 시퀀스로 정상적으로 토큰화하지만, 토큰화 직후 전체 학습 예제를 세트, 즉 영화 리뷰에 있거나 없는 토큰들의 단순한 순서 없는 "묶음"으로 변환합니다.

여기서 핵심 아이디어는 이러한 세트를 사용하여 리뷰의 각 단어에 가중치를 부여하는 매우 간단한 모델을 구축하는 것입니다. "끔찍한"이라는 단어가 있다면 (항상 그런 것은 아니지만) 나쁜 리뷰를 나타낼 가능성이 높고, "흥미진진한"이라는 단어는 좋은 리뷰를 나타낼 가능성이 높습니다. 이러한 가중치를 학습할 수 있는 작은 모델, 즉 단어 모음 모델을 구축할 수 있습니다.

예를 들어, 다음과 같은 간단한 입력 문장과 어휘가 있다고 가정해 보겠
```
"this movie made me cry"

{"[UNK]": 0, "movie": 1, "film": 2, "made": 3, "laugh": 4, "cry": 
```
우리는 이 짧은 리뷰를 다음과 같이 토큰화할 것입니다.
```
[0, 1, 3, 0, 5]
```
순서를 무시하면 이를 토큰 ID 집합으로 변환할 수 있습니다.
```
{0, 1, 3, 5}
```
마지막으로, 멀티핫 인코딩을 사용하여 해당 집합을 어휘집과 동일한 길이의 고정 크기 벡터로 변환할 수 있습니다.
```
[1, 1, 0, 1, 0, 1]
```
여기서 다섯 번째 위치의 0은 리뷰에 "웃음"이라는 단어가 없음을 의미하고, 여섯 번째 위치의 1은 "울음"이라는 단어가 있음을 의미합니다. 이처럼 간단하게 인코딩된 입력 리뷰는 모델 학습에 직접 사용할 수 있습니다.5}습니다.

#### Training a bag-of-words model

코드로 텍스트 처리를 하려면, 앞 장에서 다룬 WordTokenizer를 확장하는 것도 간단한 방법입니다. 하지만 Keras에 내장된 TextVectorization 레이어를 사용하는 것이 훨씬 더 쉽습니다. TextVectorization은 단어와 문자 토큰화를 처리하며, 레이어 출력에 대한 멀티핫 인코딩을 포함한 여러 추가 기능을 제공합니다.

Keras의 다른 전처리 레이어들과 마찬가지로 TextVectorization 레이어에도 입력 데이터로부터 레이어 상태를 학습하는 adapt() 메서드가 있습니다. TextVectorization의 경우, adapt() 메서드는 입력 데이터셋을 순회하면서 데이터셋에 대한 어휘를 실시간으로 학습합니다. 이제 이 메서드를 사용하여 입력 데이터를 토큰화하고 인코딩해 보겠습니다. 텍스트 분류 문제에 적합한 20,000개의 단어로 구성된 어휘를 구축할 것입니다.

In [ ]:
from keras import layers

max_tokens = 20_000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
)
train_ds_no_labels = train_ds.map(lambda x, y: x)
text_vectorization.adapt(train_ds_no_labels)

bag_of_words_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

전처리된 입력 데이터의 단일 배치를 살펴보겠습니다.

In [ ]:
x, y = next(bag_of_words_train_ds.as_numpy_iterator())
x.shape

In [ ]:
y.shape

전처리 과정을 거치면 배치에 있는 각 샘플이 20,000개의 숫자로 이루어진 벡터로 변환되는 것을 볼 수 있습니다. 각 숫자는 어휘 용어의 존재 여부를 나타냅니다.

다음으로, 아주 간단한 선형 모델을 학습시킬 수 있습니다. 모델 구축 코드는 나중에 다시 사용할 수 있도록 함수로 저장하겠습니다.

In [ ]:
def build_linear_classifier(max_tokens, name):
    inputs = keras.Input(shape=(max_tokens,))
    outputs = layers.Dense(1, activation="sigmoid")(inputs)
    model = keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_linear_classifier(max_tokens, "bag_of_words_classifier")

우리 모델의 요약을 살펴보겠습니다.

In [ ]:
model.summary(line_length=80)

이 모델은 아주 간단합니다. 단어 하나당 하나씩, 총 20,001개의 매개변수와 편향 용어 하나가 있을 뿐입니다. 이제 학습을 시작해 보겠습니다. 7장에서 다룬 EarlyStopping 콜백을 추가하면 검증 손실이 더 이상 개선되지 않을 때 학습이 자동으로 중지되고 가장 좋은 에포크의 가중치로 복원됩니다.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
history = model.fit(
    bag_of_words_train_ds,
    validation_data=bag_of_words_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

우리 모델은 크기를 고려하면 당연한 결과이지만, 학습 시간은 1분도 채 걸리지 않습니다. 실제로 입력 데이터의 토큰화 및 인코딩 작업이 모델 매개변수 업데이트보다 훨씬 더 많은 비용을 소모합니다. 모델 정확도를 그래프로 나타내 보겠습니다(그림 14.2).

In [ ]:
import matplotlib.pyplot as plt

accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, "r--", label="Training accuracy")
plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()

검증 성능이 크게 저하되지 않고 오히려 안정화되는 것을 볼 수 있습니다. 우리 모델은 매우 단순해서 과적합될 가능성이 거의 없습니다. 이제 테스트 세트에서 성능을 평가해 보겠습니다.

In [ ]:
test_loss, test_acc = model.evaluate(bag_of_words_test_ds)
test_acc

단일 CPU에서도 효율적으로 실행될 수 있을 만큼 가벼운 학습 작업으로도 리뷰의 감정을 88%의 정확도로 예측할 수 있습니다.

이 예시에서 단어 토큰화 방식을 선택한 점에 주목할 필요가 있습니다. 문자 수준 토큰화를 피한 이유는 명확합니다. 영화 리뷰의 모든 문자를 "묶어서" 토큰화한다고 해서 영화 내용에 대한 정보를 거의 얻을 수 없기 때문입니다. 충분히 큰 어휘를 사용한다면 서브워드 토큰화가 좋은 선택이 될 수 있지만, 이 예시에서는 그럴 필요가 거의 없습니다. 학습하는 모델의 크기가 작기 때문에 학습 속도가 빠르고 가중치가 실제 영어 단어에 해당하는 어휘를 사용하는 것이 편리합니다.

효율적인 텍스트 전처리

모든 응용 머신러닝에서 전처리의 속도와 효율성은 중요한 고려 사항입니다. 프로그램 속도가 빠를수록 좋지만, 특히 GPU와 TPU 같은 가속기의 가격이 높은 경우에는 더욱 그렇습니다. 입력 전처리를 하는 동안 값비싼 GPU가 유휴 상태로 있는 것을 피해야 합니다!

텍스트 전처리는 항상 CPU에서 실행되어야 한다는 점에서 독특합니다. GPU는 숫자 입력만 처리하므로 모든 토큰화는 GPU 학습 단계 이전에 완료되어야 합니다. 한 가지 방법은 토큰화된 입력을 미리 계산하는 것입니다. 토큰화는 모델 가중치에 의존하지 않으므로 모든 입력 텍스트 파일을 토큰화하고 학습 시작 전에 정수 시퀀스로 다시 저장할 수 있습니다. 그러나 이것이 항상 실용적인 것은 아닙니다. 실시간으로 텍스트를 토큰화하면 더 빠른 실험이 가능합니다. 이전에 보지 못한 예제에 대해 추론을 실행하는 경우 토큰화된 입력을 미리 계산할 수 없으므로 토큰화와 순방향 패스를 빠르게 연속적으로 실행해야 합니다.

실시간으로 텍스트 입력을 전처리할 때 핵심은 "충분히 빠른" 속도를 확보하는 것입니다. 고가의 GPU에 항상 새로운 전처리된 데이터가 공급되도록 해야 합니다. 그렇지 않으면 GPU가 병목 현상을 일으키고, 토큰화 속도를 개선해도 얻을 것이 없습니다.

이전 장에서 tf.data를 살펴봤는데, 이 라이브러리를 사용하는 중요한 이유는 CPU가 GPU나 TPU의 병목 현상이 되는 것을 방지하도록 설계되었기 때문입니다. 이 장 전체에서 tf.data를 사용합니다. `keras.utils.text_dataset_from_directory()`는 tf.data.Dataset을 로드하고, `map()`은 입력 데이터에 텍스트 벡터화 레이어를 적용하는 등의 변환을 수행합니다. tf.data는 여러 CPU 코어에서 텍스트 전처리를 병렬로 실행하여 학습 실행 중 가속기의 병목 현상을 방지합니다.

이 장의 코드는 여전히 멀티 백엔드를 사용하고 있다는 점에 유의해야 합니다(실제로 이 장의 출력은 Jax를 사용하여 생성했습니다). tf.data는 PyTorch, JAX 또는 TensorFlow 자체와 함께 사용할 수 있습니다. Keras는 입력 Tensor를 주어진 백엔드에 맞는 올바른 형식으로 자동으로 변환합니다.

#### Training a bigram model

물론, 단어 순서를 완전히 무시하는 것은 지나치게 단순화된 접근 방식이라는 것을 직관적으로 짐작할 수 있습니다. 왜냐하면 기본적인 개념조차 여러 단어로 표현될 수 있기 때문입니다. 예를 들어, "United States"라는 단어는 "states"와 "united"라는 단어를 각각 따로 떼어놓았을 때와는 완전히 다른 의미를 전달합니다. "나쁘지 않은" 영화와 "나쁜" 영화는 서로 다른 감정 점수를 받아야 마땅합니다.

따라서, 현재 우리가 구축하고 있는 이러한 단순한 집합 기반 모델에서도 단어 순서에 대한 정보를 모델에 반영하는 것이 일반적으로 좋은 방법입니다. 이를 위한 쉬운 방법 중 하나는 입력 텍스트에서 연속적으로 나타나는 두 개의 토큰, 즉 바이그램을 고려하는 것입니다. 예를 들어, "this movie made me cry"라는 문장에서 {"this", "movie", "made", "me", "cry"}는 입력 텍스트에 있는 모든 단어 유니그램의 집합이고, {"this movie", "movie made", "made me", "me cry"}는 모든 바이그램의 집합입니다. 방금 학습시킨 단어 모음 모델은 단일어 모델이라고도 할 수 있으며, n-gram은 임의의 n에 대해 n개의 토큰이 순서대로 나열된 시퀀스를 의미합니다.

이중어를 모델에 추가하려면 어휘를 구축할 때 모든 이중어의 빈도를 고려해야 합니다. 이는 두 가지 방법으로 구현할 수 있습니다. 이중어로만 구성된 어휘를 만들거나, 이중어와 단일어가 같은 어휘에 포함될 수 있도록 하는 것입니다. 후자의 경우, 입력 텍스트에서 "United States"가 "ventriloquism"보다 더 자주 나타난다면 "United States"가 "ventriloquism"보다 먼저 어휘에 포함됩니다.

이 또한 앞부분에서 다룬 WordTokenizer를 확장하여 구현할 수 있지만, TextVectorization은 기본적으로 이 기능을 제공합니다. 이제 이중어를 고려하여 어휘를 약간 더 크게 학습시키고, adapt() 함수를 사용하여 새로운 어휘를 생성한 다음, 이중어를 포함한 출력 벡터를 멀티핫 인코딩합니다.

In [ ]:
max_tokens = 30_000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
    ngrams=2,
)
text_vectorization.adapt(train_ds_no_labels)

bigram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

사전 처리된 입력 데이터 배치를 다시 살펴보겠습니다.

In [ ]:
x, y = next(bigram_train_ds.as_numpy_iterator())
x.shape

우리가 가진 어휘의 작은 부분을 살펴보면, 단어 하나로 이루어진 단어와 두 단어로 이루어진 단어가 모두 존재함을 알 수 있습니다.

In [ ]:
text_vectorization.get_vocabulary()[100:108]

입력 데이터에 대한 새로운 인코딩 방식을 사용하면 이전과 동일하게 선형 모델을 학습시킬 수 있습니다.

In [ ]:
model = build_linear_classifier(max_tokens, "bigram_classifier")
model.fit(
    bigram_train_ds,
    validation_data=bigram_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

이 모델은 저희의 단어 모음 모델보다 약간 더 크지만(매개변수 20,001개 대신 30,001개), 학습 시간은 거의 비슷합니다. 성능은 어땠을까요?

In [ ]:
test_loss, test_acc = model.evaluate(bigram_test_ds)
test_acc

이제 테스트 정확도가 90%에 도달했습니다. 눈에 띄는 개선입니다!

삼중어(세 단어로 이루어진 조합)를 고려하면 이 수치를 더욱 높일 수 있지만, 삼중어를 넘어서면 문제가 빠르게 해결 불가능해집니다. 영어에서 가능한 4-그램의 공간은 방대하며, 시퀀스가 ​​길어질수록 문제는 기하급수적으로 증가합니다. 4-그램을 제대로 커버하려면 엄청난 어휘가 필요하고, 모델은 단순히 가중치가 부여된 문장 조각 전체를 암기하는 데 그쳐 일반화 능력을 잃게 될 것입니다. 더 긴 순서의 텍스트 시퀀스를 안정적으로 고려하려면 더욱 발전된 모델링 기법이 필요합니다.

### Sequence models

이제 테스트 정확도가 90%에 도달했습니다. 눈에 띄는 개선입니다!

삼중어(세 단어로 이루어진 조합)를 고려하면 정확도를 더욱 높일 수 있지만, 삼중어를 넘어서면 문제가 빠르게 해결 불가능해집니다. 영어에서 가능한 4-그램의 공간은 방대하며, 시퀀스가 ​​길어질수록 문제는 기하급수적으로 커집니다. 4-그램을 제대로 커버하려면 엄청난 어휘가 필요하고, 모델은 단순히 전체 스니펫을 암기하는 데 그쳐 일반화 능력을 잃게 될 것입니다. 이전 두 모델을 통해 시퀀스 정보가 중요하다는 것을 알 수 있었습니다. 기본적인 선형 모델에 단어 순서에 대한 정보를 담은 특징을 추가하여 성능을 향상시켰습니다.

하지만 이는 입력 특징을 수동으로 설계한 결과이며, 이 접근 방식은 몇몇 단어의 국소적인 순서 정보까지만 적용할 수 있다는 것을 알 수 있습니다. 딥러닝에서 흔히 그렇듯이, 이러한 특징을 직접 구축하기보다는 모델에 원시 단어 시퀀스를 제공하고 토큰 간의 위치 의존성을 직접 학습하도록 하는 것이 좋습니다.

완전한 토큰 시퀀스를 입력으로 받는 모델을 간단히 말해 시퀀스 모델이라고 합니다. 아키텍처에는 몇 가지 선택지가 있습니다. 시계열 모델링에서처럼 RNN 모델을 구축할 수도 있고, 이미지 처리 모델과 유사하게 단일 시퀀스 차원에 필터를 컨볼루션하는 1D 컨볼루션 신경망(ConvNet)을 구축할 수도 있습니다. 그리고 다음 장에서 자세히 살펴볼 트랜스포머(Transformer) 모델을 구축할 수도 있습니다.

이러한 접근 방식을 취하기 전에 입력 데이터를 순서가 있는 시퀀스로 전처리해야 합니다. 이 장의 토큰화 부분에서 살펴본 것처럼 토큰 ID의 정수 시퀀스가 ​​필요하지만, 한 가지 추가적인 고려 사항이 있습니다. 배치 처리 방식으로 입력 데이터를 처리할 때, 모든 입력 데이터가 직사각형 형태여야 GPU에서 배치 전체에 걸쳐 효율적으로 병렬 처리가 가능합니다. 하지만 토큰화된 입력 데이터는 거의 항상 길이가 다양합니다. IMDb 영화 리뷰는 몇 문장에서 여러 단락에 이르기까지 다양하며, 단어 수도 제각각입니다.

이러한 사실을 반영하기 위해 입력 시퀀스를 잘라내거나 이전에 사용했던 "[UNK]" 토큰과 유사한 특수 토큰 "[PAD]"로 "패딩"할 수 있습니다. 예를 들어, 두 개의 입력 문장과 가중치가 부여된 문장의 원하는 길이가 8개인 경우를 생각해 보겠습니다. 더 긴 순서의 텍스트 시퀀스를 안정적으로 처리하려면 더 고급 모델링 

```
"the quick brown fox jumped over the lazy dog"

"the slow brown badge
```
우리는 다음 토큰들을 정수 ID로 토큰화할 것입니다:
```
["the", "quick", "brown", "fox", "jumped", "over", "the", "lazy"]
["the", "slow", "brown", "badger", "[PAD]", "[PAD]", "[PAD]", "[PAD]"
```
이렇게 하면 배치 계산 속도가 훨씬 빨라지지만, 패딩 토큰이 모델 예측 품질에 영향을 미치지 않도록 주의해야 합니다.

입력 크기를 적절하게 유지하기 위해 IMDb 리뷰에서 처음 600단어를 잘라낼 수 있습니다. 평균 리뷰 길이가 233단어이고 600단어보다 긴 리뷰는 5%에 불과하므로 이는 합리적인 선택입니다. 다시 한번, 입력에 패딩 또는 잘라내기 옵션을 제공하고 학습된 어휘의 인덱스 0에 "PAD"를 포함하는 TextVecotorization 레이어를 사용할 수 있습니다.]r"기법이 필요합니다.

In [ ]:
max_length = 600
max_tokens = 30_000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(train_ds_no_labels)

sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

단일 입력 배치를 살펴보겠습니다.

In [ ]:
x, y = next(sequence_test_ds.as_numpy_iterator())
x.shape

In [ ]:
x

전처리 후 각 배치는 (배치 크기, 시퀀스 길이) 형태를 가지며, 거의 모든 훈련 샘플은 끝부분에 패딩을 위해 여러 개의 0을 포함합니다.

#### Training a recurrent model

LSTM을 학습시켜 보겠습니다. 이전 장에서 살펴본 것처럼 LSTM은 시퀀스 데이터를 효율적으로 처리할 수 있습니다. 하지만 이를 적용하기 전에 토큰 ID 정수를 Dense 레이어가 처리할 수 있는 부동 소수점 데이터로 변환해야 합니다.

가장 간단한 방법은 전체 시퀀스에 적용했던 멀티핫 인코딩과 유사하게 입력 ID를 원핫 인코딩하는 것입니다. 각 토큰은 모든 요소가 0이고 어휘 목록에서 해당 토큰의 인덱스에만 1이 있는 긴 벡터가 됩니다. 이제 입력 시퀀스를 원핫 인코딩하는 레이어를 만들어 보겠습니다.

In [ ]:
from keras import ops

class OneHotEncoding(keras.Layer):
    def __init__(self, depth, **kwargs):
        super().__init__(**kwargs)
        self.depth = depth

    def call(self, inputs):
        flat_inputs = ops.reshape(ops.cast(inputs, "int"), [-1])
        one_hot_vectors = ops.eye(self.depth)
        outputs = ops.take(one_hot_vectors, flat_inputs, axis=0)
        return ops.reshape(outputs, ops.shape(inputs) + (self.depth,))

one_hot_encoding = OneHotEncoding(max_tokens)

이 레이어를 단일 입력 배치에 적용해 보겠습니다.

In [ ]:
x, y = next(sequence_train_ds.as_numpy_iterator())
one_hot_encoding(x).shape

이 레이어를 모델에 직접 통합하고 양방향 LSTM을 사용하여 토큰 시퀀스를 따라 정보가 앞뒤로 모두 전달되도록 할 수 있습니다. 나중에 생성 부분을 살펴보면 단방향 시퀀스 모델(토큰 상태가 바로 앞 토큰 상태에만 의존하는 모델)이 필요하게 될 것입니다. 분류 작업에는 양방향 LSTM이 적합합니다.

이제 모델을 구축해 보겠습니다.

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(max_length,), dtype="int32")
x = one_hot_encoding(inputs)
x = layers.Bidirectional(layers.LSTM(hidden_dim))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_one_hot")
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

모델 요약을 살펴보면 매개변수 개수를 대략적으로 파악할 수 있습니다.

In [ ]:
model.summary(line_length=80)

이는 단어 단위 모델이나 두 단어 단위 모델에 비해 규모가 상당히 커진 것입니다. 약 1,500만 개의 파라미터를 가진 이 모델은 단 하나의 LSTM 레이어만을 사용했음에도 불구하고, 지금까지 이 책에서 학습시킨 모델 중 가장 큰 모델 중 하나입니다. 이제 모델을 학습시켜 보겠습니다.

In [ ]:
# ⚠️NOTE⚠️: The following fit call will error on a T4 GPU on the TensorFlow
# backend due to a bug in TensorFlow. If you the follow cell errors out,
# do one of the following:
# - Skip the following two cells.
# - Switch to the Jax or Torch backend and re-run this notebook.
# - Change the GPU type in your runtime (requires Colab Pro as of this writing).

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

성능은 어떻습니까?

In [ ]:
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_acc

이 모델은 작동은 하지만, 특히 이전 섹션의 경량 모델과 비교했을 때 학습 속도가 매우 느립니다. 이는 입력 데이터의 크기가 상당히 크기 때문입니다. 각 입력 샘플은 (600, 30000) 크기의 행렬로 인코딩됩니다(샘플당 600단어, 가능한 단어 수 30,000개). 즉, 영화 리뷰 하나에 1,800만 개의 부동 소수점 숫자가 필요하다는 뜻입니다! 양방향 LSTM은 처리해야 할 작업이 매우 많습니다. 속도가 느릴 뿐만 아니라 테스트 정확도도 84%에 그칩니다. 이는 매우 빠른 세트 기반 모델에 비해 성능이 훨씬 떨어집니다.

단어를 벡터로 변환하기 위해 원핫 인코딩을 사용한 것은 분명 좋은 방법이 아니었습니다. 더 나은 방법이 있습니다. 바로 단어 임베딩입니다.

#### Understanding word embeddings

원핫 인코딩을 통해 데이터를 인코딩할 때는 특징 엔지니어링에 대한 결정을 내리는 것입니다. 즉, 특징 공간의 구조에 대한 근본적인 가정을 모델에 주입하는 것입니다. 그 가정은 인코딩되는 각 토큰이 서로 독립적이라는 것입니다. 실제로 원핫 벡터는 모두 서로 직교합니다. 하지만 단어의 경우, 이 가정은 명백히 틀렸습니다. 단어는 구조화된 공간을 형성하며 서로 정보를 공유합니다. 대부분의 문장에서 "movie"와 "film"은 서로 바꿔 쓸 수 있으므로 "movie"를 나타내는 벡터는 "film"을 나타내는 벡터와 직교해서는 안 됩니다. 두 벡터는 동일하거나 매우 유사해야 합니다.

더 추상적으로 말하자면, 두 단어 벡터 사이의 기하학적 관계는 이 단어들 사이의 의미적 관계를 반영해야 합니다. 예를 들어, 합리적인 단어 벡터 공간에서는 동의어가 유사한 단어 벡터에 내장될 것으로 예상할 수 있으며, 일반적으로 두 단어 벡터 사이의 기하학적 거리(코사인 거리 또는 L2 거리 등)는 관련 단어 사이의 "의미적 거리"와 상관관계가 있을 것으로 예상할 수 있습니다. 의미가 다른 단어는 서로 멀리 떨어져 있어야 하고, 관련 있는 단어는 더 가까이 있어야 합니다.

단어 임베딩은 바로 이러한 특성을 구현하는 단어의 벡터 표현입니다. 즉, 인간 언어를 구조화된 기하학적 공간으로 매핑합니다.

원핫 인코딩을 통해 얻은 벡터는 이진 벡터이고, 희소 벡터(대부분 0으로 구성됨)이며, 매우 높은 차원(어휘에 있는 단어 수와 동일한 차원)을 가지는 반면, 단어 임베딩은 저차원 부동 소수점 벡터(즉, 희소 벡터와 반대되는 밀집 벡터)입니다(그림 14.3 참조). 매우 방대한 어휘를 다룰 때는 256차원, 512차원 또는 1,024차원의 단어 임베딩을 흔히 볼 수 있습니다. 반면, 원핫 인코딩을 사용하면 현재 어휘의 경우 일반적으로 30,000차원의 벡터가 생성됩니다. 즉, 단어 임베딩은 훨씬 적은 차원에 더 많은 정보를 담을 수

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch14/word-representations.b71fcc82.png" width="300"><br>Figure 14.3: Word representations obtained from one-hot encoding or hashing are sparse, high-dimensional, and hardcoded. Word embeddings are dense, relatively low-dimensional, and learned from data.</p>

단어 임베딩은 밀집된 표현일 뿐만 아니라 구조화된 표현이기도 하며, 그 구조는 데이터로부터 학습됩니다. 유사한 단어는 가까운 위치에 임베딩되고, 임베딩 공간에서 특정 방향은 의미를 갖습니다. 이를 더 명확하게 이해하기 위해 구체적인 예를 살펴보겠습니다. 그림 14.4에서 고양이, 개, 늑대, 호랑이 네 단어가 2차원 평면에 임베딩되어 있습니다. 여기서 선택한 벡터 표현을 사용하면 이 단어들 사이의 의미적 관계를 기하학적 변환으로 인코딩할 수 있습니다. 예를 들어, 동일한 벡터를 사용하여 고양이에서 호랑이로, 개에서 늑대로 이동할 수 있습니다. 이 벡터는 "애완동물에서 야생 동물로"를 나타내는 벡터로 해석될 수 있습니다. 마찬가지로, 다른 벡터를 사용하면 개에서 고양이로, 늑대에서 호랑이로 이동할 수 있으며, 이는 "개과 동물에서 고양이과 동물로"를 나타내는 벡터로 해석될 수 있습니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch14/word-embeddings.1bc937b3.png" width="300"><br>Figure 14.4: A toy example of a word-embedding space</p>

실제 단어 임베딩 공간에서 의미 있는 기하학적 변환의 대표적인 예로는 "성별" 벡터와 "복수" 벡터가 있습니다. 예를 들어, "king" 벡터에 "여성" 벡터를 더하면 "queen" 벡터가 되고, "복수" 벡터를 더하면 "kings" 벡터가 됩니다. 단어 임베딩 공간에는 일반적으로 이처럼 해석 가능하고 잠재적으로 유용한 벡터가 수천 개 존재합니다.

이제 이러한 임베딩 공간을 실제로 어떻게 활용하는지 살펴보겠습니다. 있습니다.

#### Using a word embedding

인간 언어를 완벽하게 매핑하고 모든 자연어 처리(NPL) 작업에 사용할 수 있는 이상적인 단어 임베딩 공간이 존재할까요? 가능성은 있지만, 아직 그러한 것을 계산해낸 적은 없습니다. 또한, 우리가 매핑할 수 있는 단일한 인간 언어는 존재하지 않습니다. 언어는 매우 다양하며, 특정 문화와 맥락을 반영하기 때문에 서로 동형적이지 않습니다. 보다 현실적으로, 좋은 단어 임베딩 공간을 만드는 요소는 작업에 따라 크게 달라집니다. 영어 영화 리뷰 감정 분석 모델에 적합한 이상적인 단어 임베딩 공간은 영어 법률 문서 분류 모델에 적합한 이상적인 임베딩 공간과 다를 수 있습니다. 특정 의미 관계의 중요도가 작업마다 다르기 때문입니다.

따라서 새로운 작업마다 새로운 임베딩 공간을 학습하는 것이 합리적입니다. 다행히 역전파 알고리즘을 사용하면 이 과정이 간편해지고, Keras를 사용하면 더욱 쉬워집니다. 핵심은 Keras 임베딩 레이어의 가중치를 학습하는 것입니다.

임베딩 레이어는 정수 인덱스(특정 단어를 나타냄)를 밀집 벡터에 매핑하는 사전으로 이해하는 것이 가장 좋습니다. 정수를 입력으로 받아 내부 사전에서 해당 인덱스를 찾아 연결된 벡터를 반환합니다. 사실상 사전 검색과 같습니다(그림 14.5 

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch14/embedding-dictionary.80faa429.png" wid5h="300"><br>Figure 14.5: An Embedding layer acts as a dictionary mapping ints to floating point vectors.</p>

임베딩 레이어는 입력으로 (배치 크기, 시퀀스 길이) 형태의 랭크 2 텐서를 받습니다. 여기서 각 항목은 정수 시퀀스입니다. 레이어는 (배치 크기, 시퀀스 길이, 임베딩 크기) 형태의 부동 소수점 텐서를 반환합니다.

임베딩 레이어를 인스턴스화할 때, 다른 레이어와 마찬가지로 가중치(내부 토큰 벡터 사전)는 초기에는 무작위로 설정됩니다. 학습 과정에서 이러한 단어 벡터는 역전파를 통해 점진적으로 조정되어 하위 모델이 활용할 수 있는 구조화된 공간을 형성합니다. 학습이 완료되면 임베딩 공간은 특정 문제에 특화된 구조를 보여줍니다.

이제 임베딩 레이어를 포함하는 모델을 구축하고 해당 작업에 대한 성능을 벤치마킹해 보겠습니다.참조).

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(max_length,), dtype="int32")
x = keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim,
    mask_zero=True,
)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_embedding")
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

임베딩 레이어의 첫 번째와 두 번째 인수는 비교적 간단합니다. `input_dim`은 레이어에 입력되는 정수 값의 전체 범위를 설정합니다. 즉, 딕셔너리 조회에 사용할 수 있는 키의 총 개수를 지정합니다. `output_dim`은 조회할 출력 벡터의 차원을 설정합니다. 즉, 단어에 대한 구조화된 벡터 공간의 차원을 지정합니다.

세 번째 인수인 `mask_zero=True`는 조금 더 미묘한 의미를 가집니다. 이 인수는 Keras에게 시퀀스에서 어떤 입력이 "PAD" 토큰인지 알려주어 모델 후반부에서 이러한 항목을 마스킹할 수 있도록 합니다.

시퀀스 입력을 전처리할 때, 원래 입력에 많은 패딩 토큰을 추가하여 다음과 같은 토큰 시퀀스를 만들 수 있다는 점을 기억
```
["the", "movie", "was", "awful", "[PAD]", "[PAD]", "[PAD]", "[PAD]"]
```
모든 패딩 토큰은 먼저 임베딩된 후 LSTM 레이어에 입력됩니다. 즉, LSTM 셀에서 받는 마지막 표현에는 "PAD" 토큰 표현을 반복적으로 처리한 결과가 포함될 수 있습니다. 우리는 이전 시퀀스의 마지막 "PAD" 토큰에 대한 학습된 LSTM 표현에는 관심이 없습니다. 대신, 패딩이 없는 마지막 토큰인 "awful"의 표현에 관심이 있습니다. 다시 말해, 최종 출력 예측에 영향을 미치지 않도록 모든 "PAD" 토큰을 마스킹해야 합니다.

"mask_zero=True"는 Keras의 임베딩 레이어에서 이러한 마스킹을 쉽게 수행하는 간단한 구문입니다. Keras는 시퀀스에서 초기값이 0인 모든 요소를 ​​표시합니다. 여기서 0은 "PAD" 토큰의 토큰 ID로 간주됩니다. 이 마스크는 LSTM 레이어에서 내부적으로 사용됩니다. 전체 시퀀스에 대한 마지막 학습된 표현을 출력하는 대신, 마스킹되지 않은 마지막 표현을 출력합니다.

이 마스킹 방식은 암묵적이고 사용하기 쉽지만, 필요에 따라 시퀀스에서 마스킹할 항목을 명시적으로 지정할 수도 있습니다. LSTM 레이어는 명시적 또는 사용자 지정 마스킹을 위한 선택적 `mask` 호출 인수를 받습니다.

이 새로운 모델을 학습시키기 전에 모델 요약을 살펴보겠습니다.하세요.

In [ ]:
model.summary(line_length=80)

원핫 인코딩된 LSTM 모델의 파라미터 수를 1,500만 개에서 200만 개로 줄였습니다. 이제 모델을 학습시키고 평가해 보겠습니다.

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_acc

임베딩을 사용함으로써 학습 시간과 모델 크기를 10분의 1로 줄였습니다. 학습된 임베딩은 입력값을 원핫 인코딩하는 것보다 훨씬 효율적입니다.

하지만 LSTM의 전반적인 성능은 크게 변하지 않았습니다. 정확도는 여전히 84% 안팎에 머물렀고, bag-of-words나 바이그램 모델과는 큰 차이가 있었습니다. 그렇다면 입력 토큰에 대한 "구조화된 임베딩 공간"은 실질적으로 그다지 유용하지 않은 것일까요? 아니면 텍스트 분류 작업에는 적합하지 않은 것일까요?

오히려 그 반대입니다. 잘 학습된 토큰 임베딩 공간은 이러한 모델의 실질적인 성능 한계를 극적으로 향상시킬 수 있습니다. 이 특정 사례의 문제는 학습 환경에 있었습니다. 2만 개의 리뷰 예시 데이터로는 효과적인 단어 임베딩을 학습하기에 충분하지 않았습니다. 10번의 학습 에포크를 마친 후, 학습 데이터셋의 정확도는 99%를 넘어섰습니다. 우리 모델은 과적합되어 입력값을 암기하기 시작했는데, 심지어 주어진 작업에 대한 최적의 단어 임베딩 세트를 학습하기도 전에 이러한 현상이 나타나고 있습니다.

이런 경우 사전 학습을 활용할 수 있습니다. 단어 임베딩을 분류 작업과 함께 학습하는 대신, 긍정/부정 리뷰 레이블 없이 더 많은 데이터를 사용하여 별도로 학습할 수 있습니다. 자세히 살펴보겠습니다.

#### 텍스트 데이터 증강

컴퓨터 비전 문제에서 데이터 증강의 중요성을 살펴본 후, 텍스트에도 같은 방식을 적용할 수 있을지 궁금할 수 있습니다. 간단히 말하면, 텍스트 데이터 증강도 가능하지만 컴퓨터 비전만큼 효과적이지는 않습니다.

기본적인 텍스트 증강 기법은 입력 텍스트에 간단한 수정을 가하여 모델의 견고성을 높이는 데 도움을 줍니다. 예를 들어, 문장에서 단어를 임의로 삭제하거나 위치를 바꿀 수 있습니다. "스페인의 비는 주로 평원에 내린다"라는 문장을 "스페인의 비는 주로 평원에 내린다"로 바꾸는 것입니다. 이렇게 수정된 입력으로 모델을 학습시키면 오타나 문법 오류에 대한 모델의 견고성을 높일 수 있습니다.

하지만 이 예시는 텍스트 증강의 큰 함정을 명확하게 보여줍니다. 의도치 않게 입력 데이터의 의미를 바꾸기 쉽다는 것입니다. 고양이 사진을 자르고 회전하고 색상을 조정해도 여전히 고양이를 알아볼 수 있는 이미지 데이터와는 달리, 언어는 순서에 따라 의미가 달라지고 아주 작은 변화에도 매우 민감합니다. 단어 두 개가 바뀌면 원래 문장의 의미와 정반대가 될 수 있습니다. 일부 텍스트 증강 기법은 알려진 동의어 표에서 단어를 대체하는 방식으로 이 문제를 해결하려고 하지만, 단어의 의미를 잘못 선택하면 이 방법 역시 불안정해질 수 있습니다. 이러한 문제들 때문에 텍스트 증강 기법이 실제로 널리 사용되지 못했습니다. 일반적으로 텍스트 증강 기법에 시간을 투자하기보다는 더 많은 텍스트 샘플을 확보하는 것이 더 나은 방법입니다.

앞으로 살펴볼 생성 모델은 이러한 문제점을 해결할 수 있는 새로운 형태의 텍스트 증강 방식을 제시하고 있습니다. 일관되고 논리적인 텍스트를 생성하는 방법을 학습한 모델을 통해, 기존 입력 데이터와 유사하지만 완전히 새로운 입력값을 생성할 수 있습니다. 이는 나름의 어려움이 있지만, 데이터가 부족하고 수집하기 어려운 문제에서 텍스트 증강의 새로운 가능성을 열어줍니다.

#### Pretraining a word embedding

지난 10년간 자연어 처리(NLP) 분야의 급속한 발전은 텍스트 모델링 문제 해결에 있어 사전 학습(pretraining)이 주요 접근 방식으로 자리 잡은 시기와 맞물려 있습니다. 단순한 집합 기반 회귀 모델을 넘어 수백만, 심지어 수십억 개의 매개변수를 가진 시퀀스 모델로 넘어가면 텍스트 모델은 엄청난 양의 데이터를 요구하게 됩니다. 일반적으로 텍스트 영역에서 특정 문제에 대한 레이블이 지정된 예제를 찾는 데 한계가 있습니다.

따라서 레이블이 지정된 데이터가 필요 없는 모델 매개변수를 학습하는 비지도 학습 작업을 고안하는 것이 중요합니다. 사전 학습 데이터는 최종 작업과 유사한 도메인의 텍스트일 수도 있고, 관심 있는 언어의 임의의 텍스트일 수도 있습니다. 사전 학습을 통해 언어의 일반적인 패턴을 학습하여 최종 작업에 특화하기 전에 모델을 효과적으로 준비할 수 있습니다.

단어 임베딩은 텍스트 사전 학습의 첫 번째 큰 성공 사례 중 하나이며, 이 섹션에서는 단어 임베딩을 사전 학습하는 방법을 살펴보겠습니다. IMDb 데이터셋 준비 과정에서 무시했던 unsup/ 디렉토리를 기억하시나요? 이 데이터셋에는 25,000개의 리뷰가 추가로 포함되어 있으며, 이는 저희의 학습 데이터와 동일한 규모입니다. 모든 학습 데이터를 결합하여 비지도 학습 작업을 통해 임베딩 레이어의 파라미터를 사전 학습하는 방법을 보여드리겠습니다.

단어 임베딩을 학습하는 가장 간단한 방법 중 하나는 CBOW(Continuous Bag of Words) 모델[2]입니다. 이 모델은 데이터셋의 모든 텍스트 위에 윈도우를 슬라이딩하면서, 윈도우 바로 오른쪽과 왼쪽에 나타나는 단어들을 기반으로 누락된 단어를 지속적으로 추측하는 방식입니다(그림 14.6). 예를 들어, 주변 단어 "백"에 "돛", "파도", "돛대"가 포함되어 있다면, 가운데 단어는 "보트" 또는 "바다"일 것이라고 추측할 수

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch14/cbow.01aaf529.png" width="300"><br>Figure 14.6: The Continuous Bag of Words predicts a word based on its surrounding context with a shallow neural network.</p>

본 IMDb 분류 문제에서는 방금 학습시킨 LSTM 모델의 단어 임베딩을 "프라이밍"하는 데 관심이 있습니다. 이전에 계산해 둔 TextVectorization 어휘를 재사용할 수 있습니다. 여기서 우리가 하려는 것은 이 어휘에 있는 각 단어에 대해 64차원의 좋은 벡터를 학습하는 것입니다.

이제 입력값을 자르거나 채우지 않고 동일한 어휘를 사용하는 새로운 TextVectorization 레이어를 생성할 수 있습니다. 이 레이어의 출력 토큰은 컨텍스트 윈도우를 텍스트에 적용하여 전처리합니다. 있습니다.

In [ ]:
imdb_vocabulary = text_vectorization.get_vocabulary()
tokenize_no_padding = keras.layers.TextVectorization(
    vocabulary=imdb_vocabulary,
    split="whitespace",
    output_mode="int",
)

데이터 전처리를 위해, 학습 데이터에 윈도우를 적용하여 연속된 9개의 토큰으로 구성된 "백(bag)"을 생성합니다. 그런 다음, 가운데 단어를 레이블로 사용하고 나머지 8개의 단어를 순서가 지정되지 않은 컨텍스트로 사용하여 레이블을 예측합니다.

이를 위해 다시 한번 tf.data를 사용하여 입력값을 전처리하지만, 이 선택이 실제 모델 학습에 사용하는 백엔드를 제한하는 것은 아닙니다.

In [ ]:
import tensorflow as tf

context_size = 4
window_size = 9

def window_data(token_ids):
    num_windows = tf.maximum(tf.size(token_ids) - context_size * 2, 0)
    windows = tf.range(window_size)[None, :]
    windows = windows + tf.range(num_windows)[:, None]
    windowed_tokens = tf.gather(token_ids, windows)
    return tf.data.Dataset.from_tensor_slices(windowed_tokens)

def split_label(window):
    left = window[:context_size]
    right = window[context_size + 1 :]
    bag = tf.concat((left, right), axis=0)
    label = window[4]
    return bag, label

dataset = keras.utils.text_dataset_from_directory(
    imdb_extract_dir / "train", batch_size=None
)
dataset = dataset.map(lambda x, y: x, num_parallel_calls=8)
dataset = dataset.map(tokenize_no_padding, num_parallel_calls=8)
dataset = dataset.interleave(window_data, cycle_length=8, num_parallel_calls=8)
dataset = dataset.map(split_label, num_parallel_calls=8)

전처리 후, 컨텍스트로 사용되는 8개의 정수 토큰 ID와 하나의 토큰 ID 레이블이 쌍을 이루는 것을 확인할 수 있습니다.

이 데이터를 사용하여 학습시킬 모델은 매우 간단합니다. 모든 컨텍스트 토큰을 임베딩하는 임베딩 레이어와 컨텍스트 토큰 "백"의 평균 임베딩을 계산하는 GlobalAveragePooling1D를 사용합니다. 그런 다음, 이 평균 임베딩을 사용하여 중간 레이블 토큰의 값을 예측합니다.

이것이 전부입니다! 주변 단어 임베딩을 기반으로 단어를 잘 예측할 수 있도록 임베딩 공간을 반복적으로 개선함으로써 영화 리뷰에 사용되는 토큰의 풍부한 임베딩을 학습할 수 있습니다.

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(2 * context_size,))
cbow_embedding = layers.Embedding(
    max_tokens,
    hidden_dim,
)
x = cbow_embedding(inputs)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(max_tokens, activation="sigmoid")(x)
cbow_model = keras.Model(inputs, outputs)
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

In [ ]:
cbow_model.summary(line_length=80)

모델이 매우 단순하기 때문에 메모리 제약에 대한 걱정 없이 큰 배치 크기를 사용하여 학습 속도를 높일 수 있습니다.

또한 배치 처리된 데이터셋에 `cache()`를 호출하여 매 에포크마다 다시 계산하는 대신 전처리된 데이터셋 전체를 메모리에 저장합니다. 이는 이처럼 단순한 모델의 경우 학습보다는 전처리에서 병목 현상이 발생하기 때문입니다. 즉, CPU에서 텍스트를 토큰화하고 슬라이딩 윈도우를 계산하는 것이 GPU에서 모델 매개변수를 업데이트하는 것보다 느립니다.

이러한 경우 전처리된 결과를 메모리나 디스크에 저장하는 것이 일반적으로 좋은 방법입니다. 이후 에포크의 학습 속도가 첫 번째 에포크보다 세 배 이상 빨라지는 것을 확인할 수 있습니다. 이는 전처리된 학습 데이터를 캐시했기 때문입니다.

In [ ]:
dataset = dataset.batch(1024).cache()
cbow_model.fit(dataset, epochs=4)

학습이 끝나면, 우리는 주변 여덟 단어만을 기반으로 가운데 단어를 약 12%의 확률로 예측할 수 있습니다. 언뜻 보기에는 훌륭한 결과처럼 들리지 않을 수 있지만, 매번 3만 개의 단어 중에서 예측해야 한다는 점을 고려하면 실제로는 상당히 괜찮은 정확도입니다.

이제 이 단어 임베딩을 사용하여 LSTM 모델의 성능을 향상시켜 보겠습니다.

#### Using the pretrained embedding for classification

이제 새로운 단어 임베딩을 학습했으므로, 이를 LSTM 모델에 적용하는 것은 간단합니다. 먼저 이전과 마찬가지로 모델을 생성합니다.

In [ ]:
inputs = keras.Input(shape=(max_length,))
lstm_embedding = layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim,
    mask_zero=True,
)
x = lstm_embedding(inputs)
x = layers.Bidirectional(layers.LSTM(hidden_dim))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_cbow")

다음으로, CBOW 임베딩 레이어에서 얻은 임베딩 가중치를 LSTM 임베딩 레이어에 적용합니다. 이는 LSTM 모델의 약 200만 개에 달하는 임베딩 파라미터에 대한 새롭고 더 나은 초기화 역할을 합니다.

In [ ]:
lstm_embedding.embeddings.assign(cbow_embedding.embeddings)

이렇게 하면 평소처럼 LSTM 모델을 컴파일하고 학습시킬 수 있습니다.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

LSTM 모델을 평가해 보겠습니다.

In [ ]:
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_acc

사전 학습된 임베딩 가중치를 사용하여 LSTM 성능을 집합 기반 모델과 거의 동일한 수준으로 향상시켰습니다. 단어 단위 모델보다는 약간 더 나은 성능을 보였지만, 두 단어 단위 모델보다는 약간 뒤처졌습니다.

이러한 결과는 그동안의 노력에 비하면 다소 실망스러울 수 있습니다. 순서 정보를 포함한 전체 시퀀스를 학습하는 것이 애초에 잘못된 생각이었을까요? 문제는 최종 LSTM 모델에 필요한 데이터가 아직 부족하다는 점입니다. 이 모델은 표현력이 풍부하고 강력하기 때문에 충분한 영화 리뷰 데이터만 있다면 집합 기반 접근 방식을 쉽게 능가할 수 있지만, 모델의 성능 한계에 도달하기 위해서는 순서가 있는 데이터에 대한 학습이 훨씬 더 많이 필요합니다.

이 문제는 충분한 컴퓨팅 자원만 있다면 쉽게 해결할 수 있습니다. 다음 장에서는 트랜스포머 모델을 다룰 것입니다. 이 모델은 더 긴 토큰 시퀀스에서 의존 관계를 학습하는 데 약간 더 뛰어나지만, 가장 중요한 점은 이러한 모델은 종종 모든 단어 순서 정보를 포함한 방대한 양의 영어 텍스트로 학습된다는 것입니다. 이를 통해 모델은 언어를 지배하는 문법 패턴의 통계적 형태를 학습할 수 있습니다. 단어 순서와 관련된 이러한 통계적 패턴은 현재 LSTM 모델이 데이터 제약으로 인해 효과적으로 학습하지 못하는 이유입니다.

하지만 텍스트 분류 성능의 한계를 뛰어넘는 대규모의 고급 모델로 나아갈 때, 우리의 바이그램 모델과 같은 간단한 집합 기반 회귀 접근 방식이 비용 대비 매우 효율적이라는 점을 강조할 필요가 있습니다. 집합 기반 모델은 매우 빠르며 매개변수도 수천 개에 불과합니다. 이는 오늘날 화제의 중심이 된 수십억 개의 매개변수를 가진 대규모 언어 모델과는 큰 차이입니다.

컴퓨팅 자원이 제한적이고 정확도를 어느 정도 희생할 수 있는 환경이라면 집합 기반 모델이 가장 비용 효율적인 접근 방식이 될 수 있습니다.

## Summary

* 모든 텍스트 모델링 문제는 텍스트를 분할하고 정수 데이터로 변환하는 전처리 단계인 토큰화를 포함합니다.
* 토큰화는 표준화, 분할, 인덱싱의 세 단계로 나눌 수 있습니다. 표준화는 텍스트를 정규화하고, 분할은 텍스트를 토큰으로 나누며, 인덱싱은 각 토큰에 고유한 정수 ID를 할당합니다.
* 토큰화에는 문자 토큰화, 단어 토큰화, 부분 단어 토큰화의 세 가지 주요 유형이 있습니다. 표현력이 충분한 모델과 충분한 학습 데이터가 있다면 부분 단어 토큰화가 일반적으로 가장 효과적입니다.
* NLP 모델은 주로 입력 토큰의 순서 처리 방식에서 차이가 납니다.
  - 집합 모델은 대부분의 순서 정보를 무시하고 입력에서 토큰의 존재 여부만을 기반으로 간단하고 빠른 모델을 학습합니다. 바이그램 또는 트라이그램 모델은 두 개 또는 세 개의 연속된 토큰의 존재 여부를 고려합니다. 집합 모델은 학습 및 배포 속도가 매우 빠릅니다.
  - 순서 모델은 입력 데이터의 토큰 순서를 사용하여 학습하려고 합니다. 순차 모델은 효과적으로 학습하기 위해 많은 양의 데이터가 필요합니다.

* 임베딩은 토큰 ID를 학습된 잠재 공간으로 변환하는 효율적인 방법입니다. 임베딩은 경사 하강법을 사용하여 일반적인 방식으로 학습할 수 있습니다.
* 사전 학습은 시퀀스 모델의 데이터 요구량 문제를 해결하는 데 매우 중요합니다. 사전 학습 과정에서 비지도 학습 작업을 통해 모델은 레이블이 지정되지 않은 대량의 텍스트 데이터로부터 학습할 수 있습니다. 학습된 매개변수는 이후 작업으로 전달될 수 있습니다.